In [1]:
import os
import sys
import glob

import itertools
import pandas as pd
import numpy as np

from typing import List, Tuple, Dict
from tqdm import tqdm
from scipy.signal import savgol_filter
from sklearn.metrics import mean_squared_error

sys.path.insert(0, '../../..') # Load Parent directory to import utils
from utils.constants import Configs
from utils.helper_functions import get_smart_retail_dataset

# ==========================================================
# Global configuration for the sensor group to be filtered.
# ==========================================================
SENSOR_GROUP = "wrist" # "wrist", "elbow" ou "shoulder"

SENSOR_COORDINATES = {
    "wrist": ["xw", "yw", "zw"],
    "elbow": ["xe", "ye", "ze"],
    "shoulder": ["xs", "ys", "zs"],
    "all": [
        "xs", "ys", "zs",
        "xe", "ye", "ze",
        "xw", "yw", "zw",
    ],
}

if SENSOR_GROUP not in ("wrist", "elbow", "shoulder"):
    raise ValueError(
        f"SENSOR_GROUP inválido: {SENSOR_GROUP!r}. "
        "Escolha entre wrist, elbow ou shoulder."
    )

SENSOR_COLS = SENSOR_COORDINATES[SENSOR_GROUP]

/home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/.venv/lib/python3.13/site-packages/tslearn/bases/bases.py:16: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


In [2]:
df = get_smart_retail_dataset(wrist_only=True, sensor_group=SENSOR_GROUP)

sensor_cols_L = ['x_L', 'y_L', 'z_L']
sensor_cols_R = ['x_R', 'y_R', 'z_R']
all_sensor_cols = sensor_cols_L + sensor_cols_R

df.head()

Loading and fusing bimanual files...


,sub,repID,movID,movType,step,movLabel,vel_L,acc_L,k_L,t_L,...,acc_R,k_R,t_R,jerk_R,x_L,y_L,z_L,x_R,y_R,z_R
0,1,1,1,0,1,Pick,0.314419,0.368028,2.618244,-1.387820,...,0.020991,5.036018,0.149055,0.000000,4.099,-174.912,867.136,46.068,170.761,882.586
1,1,1,1,0,2,Pick,0.310089,0.658925,4.893958,-1.732887,...,0.037394,8.871600,1.557598,0.016403,-1.009,-179.932,866.915,46.126,170.527,882.808
2,1,1,1,0,3,Pick,0.299228,1.044781,8.587954,-2.463496,...,0.060485,14.174708,9.758232,0.023091,-4.855,-181.429,866.665,46.091,169.899,882.909
3,1,1,1,0,4,Pick,0.286628,1.055988,9.614970,-3.797109,...,0.064352,14.938425,27.112546,0.003867,-9.062,-182.366,866.570,46.260,169.568,883.152
4,1,1,1,0,5,Pick,0.275769,0.841129,7.997407,-3.507823,...,0.064668,15.091309,48.776516,0.000316,-13.354,-183.134,866.558,46.190,169.377,883.362


In [3]:
# # TODO: Sample data for testing purposes
# df = df[df['sub'].isin([1, 2])].copy()

## Filtragem

In [4]:
MAD_TO_STD_FACTOR = 1.4826 # convert MAD to standard deviation for normally distributed data

def apply_hampel_series(series: pd.Series, window_size: int = 5, n_sigmas: float = 3.0) -> np.ndarray:
    """
    Applies the Hampel filter and PCHIP interpolation to a 1D pandas Series.
    Matches the exact behavior of the original rolling window implementation.
    """
    if series.empty:
        return series.values

    k = 2 * window_size + 1
    rolling_median = series.rolling(window=k, center=True).median()
    
    # Fast vectorized MAD calculation using rolling median of absolute deviations
    # Instead of slow Python lambda inside .apply(), we compute abs deviations via numpy/scipy
    vals = series.values
    med_vals = rolling_median.values
    
    # Calculate rolling MAD using pandas rolling median on pre-centered windows where possible,
    # or keep exact rolling apply with raw=True
    rolling_mad = series.rolling(window=k, center=True).apply(
        lambda x: np.median(np.abs(x - np.median(x))), raw=True
    )

    rolling_sigma = MAD_TO_STD_FACTOR * rolling_mad
    lower_bound = rolling_median - (n_sigmas * rolling_sigma)
    upper_bound = rolling_median + (n_sigmas * rolling_sigma)

    is_outlier = (series < lower_bound) | (series > upper_bound)

    if not is_outlier.any():
        return vals

    series_clean = series.copy()
    series_clean[is_outlier] = np.nan

    # Interpolate using pchip and handle boundary NaNs safely
    interpolated = series_clean.interpolate(method='pchip').bfill().ffill()
    return interpolated.values

def apply_hampel_group_matrix(raw_matrix: np.ndarray, window_size: int = 5, n_sigmas: float = 3.0) -> np.ndarray:
    """
    Applies Hampel filter across all sensor columns of a single group matrix (N_samples x N_cols).
    """
    n_samples, n_cols = raw_matrix.shape
    clean_matrix = np.empty_like(raw_matrix, dtype=np.float64)
    
    for col_idx in range(n_cols):
        s = pd.Series(raw_matrix[:, col_idx])
        clean_matrix[:, col_idx] = apply_hampel_series(s, window_size=window_size, n_sigmas=n_sigmas)
        
    return clean_matrix

def compute_raw_jerk(raw_matrix: np.ndarray) -> np.ndarray:
    """Pre-computes the sum of absolute 3rd-order differences per column."""
    if raw_matrix.shape[0] < 4:
        return np.zeros(raw_matrix.shape[1], dtype=np.float64)
    return np.sum(np.abs(np.diff(raw_matrix, n=3, axis=0)), axis=0)

def evaluate_sg_on_cached_hampel(
    raw_groups: List[np.ndarray],
    raw_jerks: List[np.ndarray],
    hampel_groups: List[np.ndarray],
    sg_w: int,
    sg_o: int
) -> Tuple[float, float]:
    """
    Evaluates Savitzky-Golay filter across all pre-filtered Hampel matrices in 2D.
    """
    total_mse = 0.0
    total_jerk_reduction = 0.0
    valid_groups = 0

    for raw_mat, raw_jerk_vec, hampel_mat in zip(raw_groups, raw_jerks, hampel_groups):
        if raw_mat.shape[0] <= sg_w:
            continue  # Skip groups too small for the SG window

        # Vectorized Savitzky-Golay across all 6 columns simultaneously (axis=0)
        smooth_mat = savgol_filter(hampel_mat, window_length=sg_w, polyorder=sg_o, axis=0)

        # Metric 1: Signal Fidelity (Mean MSE across all columns)
        col_mses = np.mean((raw_mat - smooth_mat) ** 2, axis=0)
        group_mse = float(np.mean(col_mses))

        # Metric 2: Smoothness (Reduction in discrete Jerk across all columns)
        smooth_jerk_vec = np.sum(np.abs(np.diff(smooth_mat, n=3, axis=0)), axis=0)
        
        # Avoid division by zero where raw_jerk is 0
        jerk_reductions = np.where(
            raw_jerk_vec > 0,
            (raw_jerk_vec - smooth_jerk_vec) / raw_jerk_vec,
            0.0
        )
        group_jerk_red = float(np.mean(jerk_reductions))

        total_mse += group_mse
        total_jerk_reduction += group_jerk_red
        valid_groups += 1

    if valid_groups == 0:
        return float('inf'), 0.0

    return total_mse / valid_groups, total_jerk_reduction / valid_groups

## Grid Search Hampel and Savitsky Golay Filter

In [5]:
df_tuning_sample = df.copy()

hampel_windows = [3, 5, 7, 9, 11, 13]
sg_windows = np.linspace(5, 21, 9, dtype=int)
sg_orders = [2, 3, 4, 5]

# 1. Extract raw numpy arrays once per group (eliminates repeated pandas groupby overhead)
print("Extracting movement groups and pre-computing raw jerk...")
raw_groups: List[np.ndarray] = []
raw_jerks: List[np.ndarray] = []

for _, group in df_tuning_sample.groupby(['sub', 'repID', 'movID'], sort=False):
    mat = group[all_sensor_cols].to_numpy(dtype=np.float64)
    raw_groups.append(mat)
    raw_jerks.append(compute_raw_jerk(mat))

results = []

# Valid (sg_w, sg_o) pairs
valid_sg_params = [(int(sw), int(so)) for sw, so in itertools.product(sg_windows, sg_orders) if sw > so]

print("Starting Global Parameter Optimization...")
for hw in tqdm(hampel_windows, desc="Hampel Windows"):
    # Pre-compute Hampel ONLY ONCE per Hampel window size
    hampel_cached_groups = [
        apply_hampel_group_matrix(mat, window_size=hw) for mat in raw_groups
    ]
    
    # Evaluate all SG combinations in fast vectorized numpy
    for sw, so in valid_sg_params:
        mse, jerk_red = evaluate_sg_on_cached_hampel(
            raw_groups, raw_jerks, hampel_cached_groups, sw, so
        )
        
        composite_score = jerk_red / (mse + 1e-6)
        results.append({
            'hampel_w': hw,
            'sg_w': sw,
            'sg_order': so,
            'mse': mse,
            'jerk_reduction': jerk_red,
            'score': composite_score
        })

results_df = pd.DataFrame(results)

Extracting movement groups and pre-computing raw jerk...
Starting Global Parameter Optimization...


Hampel Windows: 100%|██████████| 6/6 [02:07<00:00, 21.26s/it]


In [6]:
min_mse = results_df['mse'].min()
max_mse = results_df['mse'].max()
mse_range = max_mse - min_mse
results_df['fidelity_score'] = (
    1.0 - ((results_df['mse'] - min_mse) / mse_range) if mse_range > 0 else 1.0
)

min_jerk = results_df['jerk_reduction'].min()
max_jerk = results_df['jerk_reduction'].max()
jerk_range = max_jerk - min_jerk
results_df['smoothness_score'] = (
    (results_df['jerk_reduction'] - min_jerk) / jerk_range if jerk_range > 0 else 1.0
)

# Final Composite Score (70% Fidelity, 30% Smoothness)
results_df['score'] = (0.7 * results_df['fidelity_score']) + (0.3 * results_df['smoothness_score'])

# Sort and display
results_df = results_df.sort_values(by='score', ascending=False)
print("Top 3 Filter Configurations:")
print(results_df[['hampel_w', 'sg_w', 'sg_order', 'mse', 'jerk_reduction', 'score']].head(3))

best_params = results_df.iloc[0]
hampel_w = int(best_params['hampel_w'])
sg_w = int(best_params['sg_w'])
sg_o = int(best_params['sg_order'])

print(f"\nOptimal Parameters -> Hampel Window: {hampel_w}, SG Window: {sg_w}, SG Order: {sg_o}")

Top 3 Filter Configurations:
    hampel_w  sg_w  sg_order       mse  jerk_reduction     score
12         3    11         3  0.098007        0.646314  0.907043
47         5    11         3  0.098020        0.646314  0.907038
82         7    11         3  0.098055        0.646310  0.907021

Optimal Parameters -> Hampel Window: 3, SG Window: 11, SG Order: 3


## Apply Hampel and Savitsky-Golay Filter in all coordinates with best hyperparameters

In [7]:
def apply_full_filter_group(group: pd.DataFrame) -> pd.DataFrame:
    """Applies Hampel and SG filter to all sensor columns of a group at once."""
    mat = group[all_sensor_cols].to_numpy(dtype=np.float64)
    if mat.shape[0] <= sg_w:
        raise ValueError(
            f"Group length ({mat.shape[0]}) must be strictly greater than SG window ({sg_w})."
        )
    hampel_mat = apply_hampel_group_matrix(mat, window_size=hampel_w)
    smooth_mat = savgol_filter(hampel_mat, window_length=sg_w, polyorder=sg_o, axis=0)
    return pd.DataFrame(smooth_mat, index=group.index, columns=smooth_cols)

smooth_cols = [f"{col}_clean_smooth" for col in all_sensor_cols]

In [8]:
print("Applying Hampel + SG filters to all sensor columns (Left and Right)...")

# Process all columns simultaneously per group and assign back aligned by index
smoothed_parts = [
    apply_full_filter_group(group)
    for _, group in tqdm(df.groupby(['sub', 'repID', 'movID'], sort=False), desc="Filtering groups")
]
df[smooth_cols] = pd.concat(smoothed_parts).sort_index()

print("Done")

Applying Hampel + SG filters to all sensor columns (Left and Right)...


Filtering groups: 100%|██████████| 945/945 [00:14<00:00, 63.95it/s] 


Done


### Clean Final Dataframe

In [9]:
kinematic_cols = ['vel_L', 'acc_L', 'k_L', 't_L', 'jerk_L', 'vel_R', 'acc_R', 'k_R', 't_R', 'jerk_R']
metadata_cols_final = ['sub', 'repID', 'movID', 'movType', 'step', 'movLabel']

df_final = df[metadata_cols_final + smooth_cols + kinematic_cols].copy()
df_final.columns = metadata_cols_final + all_sensor_cols + kinematic_cols

### Save to Csv

In [10]:
cache_path = Configs.SMART_RETAIL_FILTERED_DATA_BIMANUAL
df_final.to_csv(cache_path, index=False)
print(f"Processed bimanual data saved to {cache_path}")

Processed bimanual data saved to /home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/SmartRetail/SmartRetail_DTW/data/processed/filtered_data_bimanual.csv
